# VOIR: cache 16 actual Mage-Flow-Edit-Turbo trajectories

Run top to bottom on a Colab **L4, A100, or better GPU**. This does not use ComfyUI.

The notebook downloads 16 paired real RGB/albedo/mask images, runs the actual frozen `microsoft/Mage-Flow-Edit-Turbo` model with VOIR's standalone Beta schedule and `dpmpp_sde_gpu`, caches projected transformer states plus the complete latent/conditioning trajectory, backs it up to Google Drive, and pushes it to the GitHub branch `mage-cache-16`.

Create a Colab secret named `GITHUB_TOKEN` with Contents write access to `Apache0ne/voir`. `HF_TOKEN` is optional.

In [ ]:
from __future__ import annotations
import getpass, json, os, shutil, subprocess, sys
from pathlib import Path

REPOSITORY = "Apache0ne/voir"
CACHE_BRANCH = "mage-cache-16"
MAGE_COMMIT = "682e026b017ae8749dc22ad2e00f245418a45683"
MODEL_ID = "microsoft/Mage-Flow-Edit-Turbo"
COUNT, TRAIN_COUNT, MAX_SIZE = 16, 12, 512
LAYERS = "0,5,11,17,23"
PROJECTION_CHANNELS, PROJECTION_SEED, SEED_BASE = 64, 1337, 42000
ATTENTION_BACKEND = "sdpa"
PROMPT = "remove illumination, shadows, highlights, and reflections; output diffuse albedo only"
USE_DRIVE_BACKUP = True
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/VOIR_Mage_Cache/real16")
VOIR_DIR, MAGE_DIR = Path("/content/voir"), Path("/content/Mage")
DATASET_DIR = VOIR_DIR / "datasets/hf_olbedo_mage16"
CACHE_DIR = VOIR_DIR / "mage_cache/real16"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["MAGE_SOURCE_DIR"] = str(MAGE_DIR)
print(json.dumps({"model": MODEL_ID, "count": COUNT, "max_size": MAX_SIZE, "layers": LAYERS, "projection_channels": PROJECTION_CHANNELS, "cache_branch": CACHE_BRANCH}, indent=2))

In [ ]:
import torch
from google.colab import drive, userdata

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU.")
gpu = torch.cuda.get_device_properties(0)
print("GPU:", gpu.name, "VRAM GiB:", round(gpu.total_memory/2**30, 2), "Torch:", torch.__version__, "CUDA:", torch.version.cuda)
if gpu.total_memory < 20 * 2**30:
    raise RuntimeError("Use an L4/A100-class GPU with at least 20 GiB VRAM.")

def run(command, cwd=None, env=None):
    print("$", " ".join(str(x) for x in command))
    subprocess.run([str(x) for x in command], cwd=cwd, env=env, check=True)

def clone_or_reset(url, destination, ref):
    if not (destination / ".git").exists():
        if destination.exists(): shutil.rmtree(destination)
        run(["git", "clone", url, destination])
    run(["git", "fetch", "--all", "--prune"], cwd=destination)
    run(["git", "checkout", "--force", ref], cwd=destination)
    if ref == "main": run(["git", "reset", "--hard", "origin/main"], cwd=destination)

run([sys.executable, "-m", "pip", "install", "--upgrade", "diffusers==0.38.0", "transformers==5.5.0", "accelerate==1.13.0", "safetensors==0.8.0", "huggingface_hub>=0.34", "einops==0.8.2", "pydantic==2.12.5", "Pillow==12.3.0", "loguru==0.7.3", "torchsde==0.2.6", "scipy>=1.11"])
clone_or_reset(f"https://github.com/{REPOSITORY}.git", VOIR_DIR, "main")
clone_or_reset("https://github.com/microsoft/Mage.git", MAGE_DIR, MAGE_COMMIT)
run([sys.executable, "-m", "pip", "install", "-e", MAGE_DIR])
run([sys.executable, "-m", "pip", "install", "-e", VOIR_DIR])
from mage_flow.models.modules._attn_backend import set_attn_backend
set_attn_backend(ATTENTION_BACKEND)

if USE_DRIVE_BACKUP: drive.mount("/content/drive")
def secret(name, required=False):
    try: value = userdata.get(name) or ""
    except Exception: value = ""
    if required and not value: value = getpass.getpass(f"Enter {name}: ").strip()
    return value
HF_TOKEN, GITHUB_TOKEN = secret("HF_TOKEN"), secret("GITHUB_TOKEN", True)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
print("Pinned Mage commit:", subprocess.check_output(["git", "-C", str(MAGE_DIR), "rev-parse", "HEAD"], text=True).strip())

In [ ]:
run([sys.executable, VOIR_DIR / "scripts/download_hf_real30.py", "--dataset", "GDAOSU/Olbedo", "--split", "train_selected", "--count", COUNT, "--train-count", TRAIN_COUNT, "--output", DATASET_DIR], cwd=VOIR_DIR)
assert len(list((DATASET_DIR/"images").glob("*.png"))) == COUNT
assert len(list((DATASET_DIR/"albedo").glob("*.png"))) == COUNT
assert len(list((DATASET_DIR/"masks").glob("*.png"))) == COUNT

if USE_DRIVE_BACKUP and DRIVE_BACKUP_DIR.exists():
    print("Restoring resumable cache from Drive:", DRIVE_BACKUP_DIR)
    shutil.copytree(DRIVE_BACKUP_DIR, CACHE_DIR, dirs_exist_ok=True)

command = [sys.executable, VOIR_DIR/"scripts/cache_mage_real16.py", "--dataset-dir", DATASET_DIR, "--output-dir", CACHE_DIR, "--model", MODEL_ID, "--count", COUNT, "--max-size", MAX_SIZE, "--layers", LAYERS, "--projection-channels", PROJECTION_CHANNELS, "--projection-seed", PROJECTION_SEED, "--seed-base", SEED_BASE, "--prompt", PROMPT, "--attn-backend", ATTENTION_BACKEND]
env = os.environ.copy(); env["MAGE_SOURCE_DIR"] = str(MAGE_DIR)
process = subprocess.Popen([str(x) for x in command], cwd=VOIR_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout: print(line, end="")
if process.wait() != 0: raise RuntimeError("Mage cache capture failed")

if USE_DRIVE_BACKUP:
    DRIVE_BACKUP_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(CACHE_DIR, DRIVE_BACKUP_DIR, dirs_exist_ok=True)
    print("Drive backup updated:", DRIVE_BACKUP_DIR)

In [ ]:
from IPython.display import display
from PIL import Image
run_info = json.loads((CACHE_DIR/"run.json").read_text())
manifest = [json.loads(x) for x in (CACHE_DIR/"manifest.jsonl").read_text().splitlines() if x.strip()]
assert run_info["status"] == "complete" and len(manifest) == COUNT
assert all(x["model_evaluations"] == 7 and x["cache_bytes"] < 95*2**20 for x in manifest)
first = torch.load(CACHE_DIR/manifest[0]["cache"], map_location="cpu", weights_only=False)
state, sampler = first["reservoir_state"], first["sampler_cache"]
print(json.dumps({"items": len(manifest), "total_cache_GiB": round(run_info["total_cache_bytes"]/2**30, 3), "projected_hidden_states": list(state["features"].shape), "auxiliary_features": list(state["aux_features"].shape), "eval_latents": list(sampler["eval_latents"].shape), "eval_denoised": list(sampler["eval_denoised"].shape), "eval_velocity": list(sampler["eval_velocity"].shape), "reference_tokens": list(sampler["reference_tokens"].shape), "text_tokens": list(sampler["text_tokens"].shape), "schedule_sigmas": sampler["schedule_sigmas"].tolist(), "model_eval_sigmas": sampler["model_eval_sigmas"].tolist()}, indent=2))
display(Image.open(CACHE_DIR/manifest[0]["input"])); display(Image.open(CACHE_DIR/manifest[0]["albedo"])); display(Image.open(CACHE_DIR/manifest[0]["preview"]))

run([sys.executable, VOIR_DIR/"scripts/push_cache_to_github.py", "--repo-dir", VOIR_DIR, "--repository", REPOSITORY, "--branch", CACHE_BRANCH, "--path", "mage_cache/real16"], cwd=VOIR_DIR, env=os.environ.copy())